# ENN583 — Week 1 Practical

This practical introduces OpenCV and scikit-image, two Python libraries used throughout computer vision. By the end, you should be able to load, inspect, display, transform, filter, threshold, and save images.

Before running the notebook, make sure Jupyter is using `practicals/Week_01` as the current working folder. The example images are in the local `data` folder, so loading an image is as simple as using a relative path such as `data/kitchen.png`.

Each topic follows a **show, then do** pattern: first run and study a commented worked example, then modify the same idea in the following **Your turn** task.

Make sure you consult the OpenCV documentation to learn more about the functions we used and their parameters: https://docs.opencv.org/4.x/ 

You should feel free to experiment with the parameters and functions to see how they work.


In [ ]:
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

# Show plots directly underneath the code cell that creates them.
%matplotlib inline

# This notebook assumes it is run from the Week_01 folder.
# The images therefore live in the relative folder named "data".
print("Current working folder:", Path.cwd())


## 1. Loading and displaying an image

`cv.imread` returns a NumPy array. OpenCV stores colour channels in **BGR** order, while Matplotlib expects **RGB**.

In [ ]:
# The image file is inside Week_01/data, relative to this notebook.
image_path = "data/kitchen.png"

# OpenCV reads colour images as BGR arrays: blue, then green, then red.
img_bgr = cv.imread(image_path)

# cv.imread returns None instead of raising an exception when the path is wrong.
if img_bgr is None:
    raise FileNotFoundError(f"OpenCV could not read {image_path}. Is Jupyter running from Week_01?")

# Make one row with two plotting areas so we can compare the wrong and right display.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Matplotlib assumes RGB, so this display has swapped red and blue channels.
axes[0].imshow(img_bgr)
axes[0].set_title("BGR interpreted as RGB")

# Convert from OpenCV's BGR order into Matplotlib's RGB order before displaying.
img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
axes[1].imshow(img_rgb)
axes[1].set_title("Correct RGB display")

# Hide tick marks and axis labels; for images they are usually visual clutter.
for axis in axes:
    axis.axis("off")

# Ask Matplotlib to reduce overlaps between the two subplots.
plt.tight_layout()


## 2. Images are arrays

An OpenCV image has shape `(height, width, channels)`. Pixel coordinates are usually written as `(x, y)`, but NumPy indexing uses `[row, column]`, which means `[y, x]`.

Inspect the image dimensions, data type, value range, and centre pixel.

In [ ]:
# NumPy reports image dimensions as height, width, then colour channels.
height, width, channels = img_bgr.shape

# Integer division gives the coordinate of the middle column and middle row.
centre_x, centre_y = width // 2, height // 2

# type(...) tells us this is a NumPy array, not a special OpenCV object.
print("Type:", type(img_bgr))

# shape is useful for checking image size and whether the image has colour channels.
print("Shape (height, width, channels):", img_bgr.shape)

# dtype is usually uint8 for normal images: integer values from 0 to 255.
print("Data type:", img_bgr.dtype)

# min and max reveal the darkest and brightest values present anywhere in the image.
print("Value range:", img_bgr.min(), "to", img_bgr.max())

# NumPy indexing is [row, column], so use [y, x] for a pixel coordinate.
print("Centre pixel (BGR):", img_bgr[centre_y, centre_x])


### Your turn: inspect a different pixel

Modify the example above to print the BGR value of the pixel one quarter of the way across and one quarter of the way down the image. Also print the same pixel in RGB order.

In [ ]:
# Choose a point one quarter across the width and one quarter down the height.
sample_x = width // 4
sample_y = height // 4

# TODO: Implement this solution.
pass


### Worked example: colour channels

Split the image into blue, green, and red channels and display each as a grayscale image. Which structures are brightest in each channel? How do the colourful fruit and cups in the image appear in each channel?

In [ ]:
# cv.split returns the channels in OpenCV's B, G, R order.
blue, green, red = cv.split(img_bgr)

# Create three plotting areas: one for each single-channel image.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# zip lets us loop over the axes, image arrays, and display names together.
for axis, channel, name in zip(
    axes, (blue, green, red), ("Blue", "Green", "Red")
):
    # Single channels are 2D arrays, so display them as grayscale intensity images.
    axis.imshow(channel, cmap="gray", vmin=0, vmax=255)
    axis.set_title(f"{name} channel")
    axis.axis("off")

plt.tight_layout()


### Your turn: make a red-only image

Use the split channels from the example to construct a colour image that keeps the red channel but sets blue and green to zero. Start from `np.zeros_like(red)` and use `cv.merge`.

In [ ]:
zeros = np.zeros_like(red)

# TODO: Implement this solution.
pass


### Worked example: crop a region of interest

Use NumPy slicing to crop a region from the image. Remember that the order is `[y_start:y_end, x_start:x_end]`.

In [ ]:
# Array slicing selects rows first (y), followed by columns (x).
crop = img_rgb[100:450, 250:850]

plt.figure(figsize=(8, 4))
plt.imshow(crop)
plt.title(f"Crop shape: {crop.shape}")
plt.axis("off")


### Your turn: make a centred crop

Modify the crop example to extract a 400 × 300 pixel region centred in the image. Compute the slice bounds from `width`, `height`, and the requested crop size rather than typing fixed coordinates.

In [ ]:
crop_width, crop_height = 400, 300

# TODO: Implement this solution.
pass


## 3. Resize and geometric operations

OpenCV specifies resize dimensions as `(width, height)`, the reverse of NumPy image shape. Resize the image to half its original size while preserving its aspect ratio, then create horizontally flipped and rotated versions.

In [ ]:
# cv.resize expects its output size as (width, height).
half_size = (width // 2, height // 2)
img_half = cv.resize(img_rgb, half_size, interpolation=cv.INTER_AREA)
# A flip code of 1 mirrors the image horizontally.
img_flipped = cv.flip(img_half, 1)
img_rotated = cv.rotate(img_half, cv.ROTATE_90_CLOCKWISE)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, image, title in zip(
    axes,
    (img_half, img_flipped, img_rotated),
    ("Half size", "Horizontal flip", "90° clockwise"),
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()


### Your turn: resize to a target width

Adapt the half-size example to produce an image exactly 500 pixels wide while preserving its aspect ratio. Then rotate the result 90° counter-clockwise.

In [ ]:
target_width = 500

# TODO: Implement this solution.
pass


## 4. Drawing and annotation

OpenCV drawing functions modify their input array in place. Make a copy, then add a line, rectangle, circle, and text. Drawing colours are supplied in BGR order.

In [ ]:
# Drawing functions change the supplied array, so preserve the original.
annotated = img_bgr.copy()
# Coordinates are (x, y), colours are BGR, and the last value is thickness.
cv.line(annotated, (50, 50), (350, 200), (255, 0, 0), 5)
cv.rectangle(annotated, (400, 100), (800, 400), (0, 255, 0), 5)
cv.circle(annotated, (width // 2, height // 2), 80, (0, 0, 255), 5)
cv.putText(
    annotated,
    "ENN583",
    (50, height - 50),
    cv.FONT_HERSHEY_SIMPLEX,
    2,
    (0, 255, 255),
    4,
    cv.LINE_AA,
)

plt.figure(figsize=(10, 6))
plt.imshow(cv.cvtColor(annotated, cv.COLOR_BGR2RGB))
plt.axis("off")


### Matplotlib point overlays

Later in the unit, vision algorithms will return image locations such as interest points. A useful debugging habit is to overlay those `(x, y)` coordinates using Matplotlib's `scatter` function.

Unlike the OpenCV functions above, Matplotlib only adds markers to the displayed figure. It does **not** change the pixels stored in `img_bgr` or `img_rgb`.

In [ ]:
# Example image coordinates that we want to highlight.
example_points = [(180, 180), (320, 280), (650, 220), (900, 450)]

# Separate the list of (x, y) tuples into arrays of x and y coordinates.
point_x = [point[0] for point in example_points]
point_y = [point[1] for point in example_points]

fig, axis = plt.subplots(figsize=(10, 6))
axis.imshow(img_rgb)

# scatter overlays markers using image coordinates. The first two are dots.
axis.scatter(
    point_x[:2], point_y[:2], s=100, marker="o", color="magenta",
    edgecolors="white", linewidths=1.5, label="dots",
)
# marker="x" draws crosses without manually constructing them from lines.
axis.scatter(
    point_x[2:], point_y[2:], s=140, marker="x", color="green",
    linewidths=3, label="crosses",
)
axis.legend()
axis.axis("off")


### Your turn: create your own annotation

Modify the drawing example: place a rectangle around the central quarter of the image and add a short label above it. Then use `axis.scatter` to overlay the five diagonal points as cyan crosses. Increase the marker size and add a legend.

In [ ]:
your_annotation = img_bgr.copy()
top_left = (width // 4, height // 4)
bottom_right = (3 * width // 4, 3 * height // 4)
diagonal_points = [
    (
        round(top_left[0] + fraction * (bottom_right[0] - top_left[0])),
        round(top_left[1] + fraction * (bottom_right[1] - top_left[1])),
    )
    for fraction in np.linspace(0, 1, 5)
]

# TODO: Implement this solution.
pass


## 5. Brightness, contrast, and `uint8` arithmetic

Most OpenCV images use unsigned 8-bit integers, with values from 0 to 255. NumPy arithmetic can wrap around at those limits, whereas OpenCV arithmetic saturates safely.

Compare `pixel + 20` with `cv.add(pixel, 20)`, then create brighter and higher-contrast images.

In [ ]:
# uint8 values wrap around in ordinary NumPy arithmetic.
pixel = np.array([250], dtype=np.uint8)
print("NumPy uint8 addition:", pixel + 20)
print("OpenCV saturated addition:", cv.add(pixel, 20))

# cv.add saturates at 255 instead of wrapping back to zero.
brighter_bgr = cv.add(img_bgr, np.full_like(img_bgr, 40))
higher_contrast_bgr = cv.convertScaleAbs(img_bgr, alpha=1.4, beta=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, image, title in zip(
    axes,
    (img_bgr, brighter_bgr, higher_contrast_bgr),
    ("Original", "Brightness +40", "Contrast ×1.4"),
):
    axis.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()


### Your turn: make two darker images

Modify the brightness example to subtract 60 safely from every pixel. Then create a second version with contrast `alpha=0.7` and brightness offset `beta=20`. Compare both with the original.

In [ ]:
# TODO: Implement this solution.
pass


## 6. Grayscale images and histograms

Convert the image to grayscale and plot its intensity histogram. A histogram counts how many pixels have each intensity and gives a compact summary of the brightness distribution.

In [ ]:
# Grayscale images have one intensity value per pixel instead of three channels.
img_grey = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

# calcHist receives a list of images, the channel index, an optional mask,
# the number of bins, and the intensity range covered by those bins.
grey_histogram = cv.calcHist([img_grey], [0], None, [256], [0, 256])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img_grey, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[0].axis("off")
axes[1].plot(grey_histogram)
axes[1].set_title("Grayscale intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
plt.tight_layout()


### Your turn: compare colour-channel histograms

Modify the histogram example to plot separate blue, green, and red histograms on the same axes. Use the BGR channel indices `0`, `1`, and `2`, and draw each curve using its matching display colour.

In [ ]:
# TODO: Implement this solution.
pass


## 7. Smoothing and image gradients

Apply Gaussian and median smoothing. Then use the Sobel operator to estimate horizontal and vertical intensity gradients. Compare gradients from the original grayscale image with gradients from a blurred image.

Why might smoothing be useful before calculating a gradient?

In [ ]:
# Both filters smooth an image, but use different neighbourhood statistics.
gaussian = cv.GaussianBlur(img_grey, (11, 11), 0)
median = cv.medianBlur(img_grey, 11)

# CV_64F preserves negative gradients; dx and dy select the derivative axis.
sobel_x = cv.Sobel(img_grey, cv.CV_64F, 1, 0, ksize=3)
sobel_y = cv.Sobel(img_grey, cv.CV_64F, 0, 1, ksize=3)
blurred_sobel_x = cv.Sobel(gaussian, cv.CV_64F, 1, 0, ksize=3)
blurred_sobel_y = cv.Sobel(gaussian, cv.CV_64F, 0, 1, ksize=3)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
images = (
    gaussian,
    np.abs(sobel_x),
    np.abs(sobel_y),
    median,
    np.abs(blurred_sobel_x),
    np.abs(blurred_sobel_y),
)
titles = (
    "Gaussian blur",
    "Sobel x",
    "Sobel y",
    "Median blur",
    "Blurred then Sobel x",
    "Blurred then Sobel y",
)
for axis, image, title in zip(axes.flat, images, titles):
    axis.imshow(image, cmap="gray")
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()


### Your turn: combine both gradient directions

Repeat the Gaussian-plus-Sobel pipeline with a smaller 5 × 5 Gaussian kernel. Combine the horizontal and vertical gradients with `cv.magnitude`, then compare the result with the unblurred gradient magnitude.

In [ ]:
# TODO: Implement this solution.
pass


### Worked example: Difference of Gaussians

Blur the grayscale image at two scales and subtract the results. Convert to a signed or floating-point type before subtraction so negative differences are preserved.

In [ ]:
# Convert to float before subtracting so negative differences are retained.
g1 = cv.GaussianBlur(img_grey, (3, 3), 0).astype(np.float32)
g2 = cv.GaussianBlur(img_grey, (9, 9), 0).astype(np.float32)
difference_of_gaussians = g1 - g2

limit = np.max(np.abs(difference_of_gaussians))
plt.figure(figsize=(10, 5))
plt.imshow(difference_of_gaussians, cmap="coolwarm", vmin=-limit, vmax=limit)
plt.colorbar(label="Intensity difference")
plt.title("Difference of Gaussians")
plt.axis("off")


## 8. Binary thresholding

Create one binary image using a fixed threshold and another using Otsu's automatic threshold selection. Compare the masks and the threshold selected by Otsu's method.

In [ ]:
# cv.threshold returns both the threshold used and the binary image.
fixed_value, fixed_mask = cv.threshold(img_grey, 100, 255, cv.THRESH_BINARY)
# Otsu's method chooses a threshold from the image histogram.
otsu_value, otsu_mask = cv.threshold(
    img_grey, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, image, title in zip(
    axes,
    (img_grey, fixed_mask, otsu_mask),
    ("Grayscale", f"Fixed threshold: {fixed_value:.0f}", f"Otsu: {otsu_value:.0f}"),
):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()


### Your turn: invert the threshold

Modify the Otsu example to produce an inverted binary mask. Blur `img_grey` first with a 5 × 5 Gaussian filter and use `THRESH_BINARY_INV + THRESH_OTSU`.

In [ ]:
# TODO: Implement this solution.
pass


## 9. Colour segmentation in HSV

Use `kitchen.png`, convert it to HSV, and use `cv.inRange` to find red pixels. Red wraps around the hue axis, so two hue ranges are required. Apply the resulting mask with `cv.bitwise_and`.

Experiment with the bounds. Can you create useful masks for green and blue objects too?

In [ ]:
# Load the large photograph again so this section can be run independently.
kitchen_path = "data/kitchen.png"
kitchen_bgr = cv.imread(kitchen_path)
if kitchen_bgr is None:
    raise FileNotFoundError(f"OpenCV could not read {kitchen_path}. Is Jupyter running from Week_01?")

# Keep the large photo manageable in the notebook by shrinking wide images.
# If the image is already 1200 pixels wide or smaller, scale stays at 1.0.
scale = min(1.0, 1200 / kitchen_bgr.shape[1])
kitchen_bgr = cv.resize(
    kitchen_bgr,
    (0, 0),  # (0, 0) tells OpenCV to compute the size from fx and fy.
    fx=scale,
    fy=scale,
    interpolation=cv.INTER_AREA,
)

# HSV separates hue from brightness, which makes colour masking easier than RGB/BGR.
kitchen_hsv = cv.cvtColor(kitchen_bgr, cv.COLOR_BGR2HSV)

# Red lies at both ends of OpenCV's hue range, so create two masks.
red_low = cv.inRange(kitchen_hsv, (0, 80, 50), (10, 255, 255))
red_high = cv.inRange(kitchen_hsv, (170, 80, 50), (179, 255, 255))

# Combine both hue ranges, then retain pixels selected by the combined mask.
red_mask = cv.bitwise_or(red_low, red_high)
red_objects = cv.bitwise_and(kitchen_bgr, kitchen_bgr, mask=red_mask)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Convert BGR to RGB only at display time; keep processing in OpenCV's BGR format.
axes[0].imshow(cv.cvtColor(kitchen_bgr, cv.COLOR_BGR2RGB))
axes[0].set_title("Kitchen image")
axes[1].imshow(red_mask, cmap="gray")
axes[1].set_title("Red mask")
axes[2].imshow(cv.cvtColor(red_objects, cv.COLOR_BGR2RGB))
axes[2].set_title("Masked result")

for axis in axes:
    axis.axis("off")
plt.tight_layout()


### Your turn: segment a different colour

Modify the HSV example to isolate green pixels instead of red. Green occupies one continuous hue range, so only one call to `cv.inRange` is needed. Adjust the bounds if the first result misses relevant pixels.

In [ ]:
# TODO: Implement this solution.
pass


## 10. Saving and loading multiple images

Use `Path("data").glob(...)` to find PNG and JPEG images in the local `data` folder, load them, and display them side by side. Then use `cv.imwrite` to save one processed result. Remember that OpenCV expects BGR when writing a colour image.


In [ ]:
# The data folder is relative to Week_01, because that is where this notebook is run.
data_folder = Path("data")

# pathlib can collect multiple filename extensions without the glob module.
# The * operator unpacks each list of paths into one combined list.
image_paths = sorted([
    *data_folder.glob("*.png"),
    *data_folder.glob("*.jpg"),
    *data_folder.glob("*.jpeg"),
])

# Make one subplot per image. The width scales with the number of images found.
fig, axes = plt.subplots(1, len(image_paths), figsize=(7 * len(image_paths), 5))

# If there is only one image, Matplotlib returns one axis instead of a list.
# np.atleast_1d makes the later loop work in both cases.
axes = np.atleast_1d(axes)

for axis, path in zip(axes, image_paths):
    # cv.imread expects a string filename, so convert the Path object with str(...).
    image = cv.imread(str(path))
    if image is None:
        raise RuntimeError(f"Could not read {path}")

    # Convert BGR to RGB before displaying with Matplotlib.
    axis.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    axis.set_title(path.name)
    axis.axis("off")

plt.tight_layout()

# Uncomment to save the annotated image beside this notebook.
# cv.imwrite("annotated_qut.jpg", annotated)


## 11. Working with scikit-image

Scikit-image usually loads colour images directly in RGB order. Its transform functions also use `(height, width)` rather than OpenCV's `(width, height)`.

In [ ]:
import skimage as ski

# scikit-image can read directly from a string or Path, and returns RGB by default.
ski_image = ski.io.imread("data/kitchen.png")

# Resize to half the original height and half the original width.
# scikit-image uses (rows, columns), which correspond to (height, width).
ski_small = ski.transform.resize(
    ski_image,
    (ski_image.shape[0] // 2, ski_image.shape[1] // 2),
)

print("Type:", type(ski_image))
print("Original dtype and range:", ski_image.dtype, ski_image.min(), ski_image.max())
print("Resized dtype and range:", ski_small.dtype, ski_small.min(), ski_small.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(ski_image)
axes[0].set_title("Loaded with scikit-image")
axes[1].imshow(ski_small)
axes[1].set_title("Resized with scikit-image")
for axis in axes:
    axis.axis("off")
plt.tight_layout()


### Worked example: compare the libraries

Repeat grayscale conversion and Gaussian smoothing with scikit-image. Compare colour order, data types, value ranges, function arguments, and results with OpenCV.

In [ ]:
# scikit-image receives RGB directly and commonly represents intensities as floats.
ski_grey = ski.color.rgb2gray(ski_image)
ski_blurred = ski.filters.gaussian(ski_grey, sigma=3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(ski_grey, cmap="gray")
axes[0].set_title(f"Grayscale: {ski_grey.dtype}, [{ski_grey.min():.2f}, {ski_grey.max():.2f}]")
axes[1].imshow(ski_blurred, cmap="gray")
axes[1].set_title("Gaussian blur, sigma=3")
for axis in axes:
    axis.axis("off")
plt.tight_layout()


### Your turn: change the scikit-image filter

Modify the example to use `sigma=8`. Display the `sigma=3` and `sigma=8` results side by side and describe how increasing sigma changes the image.

In [ ]:
# TODO: Implement this solution.
pass


## 12. Which library is faster?

Use `%timeit` to compare equivalent OpenCV and scikit-image operations. Timing should exclude image loading and plotting. Be careful: the two libraries may use different parameter conventions, data types, and boundary handling, so an exact numerical comparison is not always straightforward.

In [ ]:
# Time only the operation itself, not file loading or plotting.
# %timeit runs a line many times and reports a stable average runtime.
print("OpenCV resize:")
%timeit cv.resize(img_rgb, (width // 2, height // 2), interpolation=cv.INTER_AREA)

print("scikit-image resize:")
%timeit ski.transform.resize(ski_image, (height // 2, width // 2))

print("OpenCV Gaussian blur:")
%timeit cv.GaussianBlur(img_grey, (11, 11), 0)

print("scikit-image Gaussian blur:")
%timeit ski.filters.gaussian(ski_grey, sigma=3)


## Optional extension

Choose one of your own photographs and build a short processing pipeline. For example:

- resize it while preserving aspect ratio;
- adjust brightness or contrast;
- construct a colour or intensity mask;
- annotate and save the result.

Briefly explain each design choice and note any case where the method does not work well.